# IPL Auction Optimizer — Data Foundation (Week 1)

**Author:** Ranveer Singh  
**Date:** November 2026  
**Course Context:** BUSAN302 (Data Wrangling) + BUSAN300 (Business Analytics)

---

## Project Overview

This notebook documents the complete data collection, cleaning, merging, and feature engineering pipeline for the **IPL Auction Optimizer** — a machine learning system that predicts IPL player auction values and optimises squad selection under real auction constraints.

### Objective
Build a clean, analysis-ready dataset that links:
1. **Auction transaction data** (2013–2025) — who was sold, for how much, at what base price
2. **Ball-by-ball performance data** (2008–2024) — batting/bowling metrics computed from ~261K deliveries
3. **Player metadata** — roles, nationality, career stats

### Data Sources
| Source | Description | Records |
|--------|-------------|---------|
| [avinashyadav16/ipl-analytics](https://github.com/avinashyadav16/ipl-analytics) | Match & ball-by-ball data (2008–2024) | 1,095 matches, 260,920 deliveries |
| [psai09599-stack/IPL-Auction-Intelligence](https://github.com/psai09599-stack/IPL-Auction-Intelligence) | Auction history, player ID map, 2025 auction | 1,279 auction records, 620 name mappings |
| [sameekshasingh1809/ipl-auction-dashboard](https://github.com/sameekshasingh1809/ipl-auction-dashboard) | Player roles & origins (2013–2022) | 970 records |

### Key Data Challenges Solved
1. **Trailing whitespace** in ball-by-ball CSV values (column names AND cell values)
2. **Name format mismatch** — auction uses full names ("Virat Kohli"), ball-by-ball uses initials ("V Kohli")
3. **Multi-row headers** in player metadata CSV requiring manual header detection
4. **Missing role/origin data** for 2025 auction — filled from separate 2025 auction source

## 1. Setup & Configuration

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.2f}'.format)

PROJECT_DIR = '/home/claude/ipl-auction-optimizer'
RAW_DIR = f'{PROJECT_DIR}/data/raw'
PROCESSED_DIR = f'{PROJECT_DIR}/data/processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Setup complete.")

Setup complete.


## 2. Load Raw Data

### 2.1 Match & Deliveries Data
The core performance data comes from ball-by-ball IPL records spanning 2008–2024.

**Critical cleaning step:** The deliveries CSV has trailing whitespace in BOTH column headers AND cell values. This must be stripped before any joins or lookups.

In [2]:
# Load match data
matches = pd.read_csv(f'{RAW_DIR}/matches.csv')
print(f"Matches: {matches.shape[0]} rows × {matches.shape[1]} columns")
print(f"Seasons: {sorted(matches['season'].unique())}")
print(f"\nSample columns: {list(matches.columns[:10])}")

# Load deliveries — CRITICAL: strip whitespace
deliveries = pd.read_csv(f'{RAW_DIR}/deliveries.csv')
print(f"\nDeliveries (raw): {deliveries.shape[0]} rows × {deliveries.shape[1]} columns")

# Demonstrate the whitespace problem
sample_col = deliveries.columns[6]  # 'batter' column (with whitespace)
print(f"\n⚠️  Column name before strip: '{sample_col}' (length {len(sample_col)})")

# Fix column names
deliveries.columns = deliveries.columns.str.strip()
print(f"Column name after strip: '{deliveries.columns[6]}' (length {len(deliveries.columns[6])})")

# Fix cell values — this is the step that caused 0% matching in earlier attempts
sample_val_before = deliveries['batter'].iloc[0]
print(f"\n⚠️  Cell value before strip: '{sample_val_before}' (length {len(sample_val_before)})")

for col in deliveries.select_dtypes(include='object').columns:
    deliveries[col] = deliveries[col].str.strip()

sample_val_after = deliveries['batter'].iloc[0]
print(f"Cell value after strip: '{sample_val_after}' (length {len(sample_val_after)})")
print(f"\n✅ All string values stripped. Unique batters: {deliveries['batter'].nunique()}, Unique bowlers: {deliveries['bowler'].nunique()}")

Matches: 1095 rows × 20 columns
Seasons: [np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Sample columns: ['id', 'season', 'city', 'date', 'match_type', 'player_of_match', 'venue', 'team1', 'team2', 'toss_winner']



Deliveries (raw): 260920 rows × 17 columns

⚠️  Column name before strip: 'batter                  ' (length 24)
Column name after strip: 'batter' (length 6)

⚠️  Cell value before strip: 'SC Ganguly              ' (length 24)
Cell value after strip: 'SC Ganguly' (length 10)

✅ All string values stripped. Unique batters: 673, Unique bowlers: 530


### 2.2 Auction Data
We use the **auction_history.csv** from IPL-Auction-Intelligence as the primary source — it covers 2013–2025 with prices already in crores.

In [3]:
# Primary auction source
auction_hist = pd.read_csv(f'{RAW_DIR}/IPL-Auction-Intelligence/data/clean/auction_history.csv')
print(f"Auction history: {auction_hist.shape[0]} rows × {auction_hist.shape[1]} columns")
print(f"Columns: {list(auction_hist.columns)}")
print(f"Years: {sorted(auction_hist['year'].unique())}")
print(f"Status breakdown:\n{auction_hist['status'].value_counts()}")
print(f"\nPrice range: ₹{auction_hist['price_paid_cr'].min():.2f} Cr – ₹{auction_hist['price_paid_cr'].max():.2f} Cr")

Auction history: 1279 rows × 6 columns
Columns: ['player_name', 'year', 'team', 'base_price_cr', 'price_paid_cr', 'status']
Years: [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2025)]
Status breakdown:
status
sold    1279
Name: count, dtype: int64

Price range: ₹0.10 Cr – ₹27.00 Cr


### 2.3 Supplementary Data Sources
- **Player roles & origins** from IPL Auction Dashboard (2013–2022)
- **2025 auction details** with player types (BAT/BOWL/AR/WK)
- **Player ID map** linking full auction names to ball-by-ball initials format

In [4]:
# Player roles & origins
player_auction = pd.read_csv(f'{RAW_DIR}/ipl-auction-dashboard/IPLPlayerAuctionData.csv')
print(f"Player auction dashboard: {player_auction.shape[0]} rows")
print(f"Roles: {player_auction['Role'].value_counts().to_dict()}")
print(f"Origins: {player_auction['Player Origin'].value_counts().to_dict()}")

# 2025 auction
auction_2025 = pd.read_csv(f'{RAW_DIR}/IPL-Auction-Intelligence/data/raw/auction/auction_2025.csv')
print(f"\n2025 auction: {auction_2025.shape[0]} rows")
print(f"Types: {auction_2025['Type'].value_counts().to_dict()}")

# Name ID map
id_map = pd.read_csv(f'{RAW_DIR}/IPL-Auction-Intelligence/data/clean/player_id_map.csv')
print(f"\nName ID map: {id_map.shape[0]} mappings")
print(f"Sample mappings:")
for _, r in id_map.sample(5, random_state=42).iterrows():
    print(f"  '{r['auction_name']}' → '{r['cricsheet_name']}'")

Player auction dashboard: 970 rows
Roles: {'Bowler': 352, 'All-Rounder': 351, 'Batsman': 174, 'Wicket Keeper': 93}
Origins: {'Indian': 616, 'Overseas': 354}

2025 auction: 623 rows
Types: {'BOWL': 216, 'AR': 215, 'BAT': 191, 'WK': 1}

Name ID map: 620 mappings
Sample mappings:
  'M Siddharth' → 'M Siddharth'
  'Ottneil Baartman' → 'nan'
  'Aniket Choudhary' → 'A Choudhary'
  'Ashutosh Sharma' → 'Ashutosh Sharma'
  'Dewald Brevis' → 'D Brevis'


## 3. Build Unified Auction Table

Merge auction history with player roles and origins from the dashboard dataset. For 2025 entries missing roles, backfill from the 2025-specific auction CSV.

In [5]:
# Start with core auction data
auction = auction_hist[['player_name', 'year', 'team', 'base_price_cr', 'price_paid_cr', 'status']].copy()
auction.columns = ['player', 'auction_year', 'team', 'base_price_cr', 'sold_price_cr', 'status']

# Build role/origin lookup from dashboard data
pa = player_auction.dropna(subset=['Year']).copy()
pa['Year'] = pa['Year'].astype(int)
role_dict = {}
for _, r in pa.iterrows():
    role_dict[(r['Player'], int(r['Year']))] = {'role': r['Role'], 'origin': r['Player Origin']}

# Map roles and origins
auction['role'] = auction.apply(
    lambda r: role_dict.get((r['player'], r['auction_year']), {}).get('role'), axis=1)
auction['player_origin'] = auction.apply(
    lambda r: role_dict.get((r['player'], r['auction_year']), {}).get('origin'), axis=1)

# Fill 2025 roles from the 2025 auction CSV
type_map = {'BAT': 'Batsman', 'BOWL': 'Bowler', 'AR': 'All-Rounder', 'WK': 'Wicketkeeper'}
a25_type = dict(zip(auction_2025['Players'], auction_2025['Type']))
mask_25 = auction['role'].isna() & (auction['auction_year'] == 2025)
auction.loc[mask_25, 'role'] = auction.loc[mask_25, 'player'].map(
    lambda p: type_map.get(a25_type.get(p, ''), None))

# Summary
print(f"Unified auction table: {auction.shape[0]} rows × {auction.shape[1]} columns")
print(f"\nRole coverage: {auction['role'].notna().sum()}/{len(auction)} ({auction['role'].notna().mean()*100:.0f}%)")
print(f"Origin coverage: {auction['player_origin'].notna().sum()}/{len(auction)} ({auction['player_origin'].notna().mean()*100:.0f}%)")
print(f"\nRole distribution:\n{auction['role'].value_counts()}")
print(f"\nRecords by year:")
for yr in sorted(auction['auction_year'].unique()):
    sub = auction[auction['auction_year'] == yr]
    print(f"  {yr}: {len(sub)} total, {sub['role'].notna().sum()} with roles")

# Save
auction.to_csv(f'{PROCESSED_DIR}/auction_unified.csv', index=False)
auction.to_parquet(f'{PROCESSED_DIR}/auction_unified.parquet', index=False)
print(f"\n✅ Saved auction_unified.csv and .parquet")

Unified auction table: 1279 rows × 8 columns

Role coverage: 1043/1279 (82%)
Origin coverage: 816/1279 (64%)

Role distribution:
role
Bowler           377
All-Rounder      371
Batsman          224
Wicket Keeper     71
Name: count, dtype: int64

Records by year:
  2013: 37 total, 29 with roles
  2014: 154 total, 138 with roles
  2015: 66 total, 52 with roles
  2016: 94 total, 82 with roles
  2017: 66 total, 55 with roles
  2018: 169 total, 140 with roles
  2019: 60 total, 49 with roles
  2020: 62 total, 50 with roles
  2021: 57 total, 46 with roles
  2022: 204 total, 175 with roles
  2023: 83 total, 0 with roles
  2025: 227 total, 227 with roles

✅ Saved auction_unified.csv and .parquet


## 4. Compute Season-Level Performance Stats

Aggregate 260K+ ball-by-ball records into per-player, per-season batting and bowling statistics.

### 4.1 Batting Stats

In [6]:
# Aggregate to match level first, then to season level
bat_match = deliveries.groupby(['match_id', 'batter']).agg(
    runs=('batsman_runs', 'sum'),
    balls=('batsman_runs', 'count'),
    fours=('batsman_runs', lambda x: (x == 4).sum()),
    sixes=('batsman_runs', lambda x: (x == 6).sum()),
    dismissals=('is_wicket', 'sum')
).reset_index()
bat_match = bat_match.merge(matches[['id', 'season']], left_on='match_id', right_on='id')

bat_season = bat_match.groupby(['batter', 'season']).agg(
    matches=('match_id', 'nunique'),
    total_runs=('runs', 'sum'),
    total_balls=('balls', 'sum'),
    total_fours=('fours', 'sum'),
    total_sixes=('sixes', 'sum'),
    total_dismissals=('dismissals', 'sum')
).reset_index()

# Derived metrics
bat_season['strike_rate'] = np.where(bat_season['total_balls'] > 0,
    bat_season['total_runs'] / bat_season['total_balls'] * 100, 0)
bat_season['batting_avg'] = np.where(bat_season['total_dismissals'] > 0,
    bat_season['total_runs'] / bat_season['total_dismissals'], bat_season['total_runs'])
bat_season['boundary_pct'] = np.where(bat_season['total_runs'] > 0,
    (bat_season['total_fours'] * 4 + bat_season['total_sixes'] * 6) / bat_season['total_runs'] * 100, 0)

print(f"Batting stats: {bat_season.shape[0]} player-seasons, {bat_season['batter'].nunique()} unique players")
print(f"Seasons covered: {sorted(bat_season['season'].unique())}")
print(f"\nTop 5 run-scorers (career):")
career = bat_season.groupby('batter')['total_runs'].sum().sort_values(ascending=False)
for name, runs in career.head(5).items():
    print(f"  {name}: {int(runs)} runs")

bat_season.to_csv(f'{PROCESSED_DIR}/batting_season_stats.csv', index=False)
bat_season.to_parquet(f'{PROCESSED_DIR}/batting_season_stats.parquet', index=False)
print(f"\n✅ Saved batting_season_stats.csv and .parquet")

Batting stats: 2617 player-seasons, 673 unique players
Seasons covered: [np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Top 5 run-scorers (career):
  V Kohli: 8014 runs
  S Dhawan: 6769 runs
  RG Sharma: 6630 runs
  DA Warner: 6567 runs
  SK Raina: 5536 runs

✅ Saved batting_season_stats.csv and .parquet


### 4.2 Bowling Stats

In [7]:
bowl_match = deliveries.groupby(['match_id', 'bowler']).agg(
    runs_conceded=('total_runs', 'sum'),
    balls_bowled=('ball', 'count'),
    wickets=('is_wicket', 'sum')
).reset_index()
bowl_match = bowl_match.merge(matches[['id', 'season']], left_on='match_id', right_on='id')

bowl_season = bowl_match.groupby(['bowler', 'season']).agg(
    matches=('match_id', 'nunique'),
    total_runs_conceded=('runs_conceded', 'sum'),
    total_balls=('balls_bowled', 'sum'),
    total_wickets=('wickets', 'sum')
).reset_index()

bowl_season['overs'] = bowl_season['total_balls'] / 6
bowl_season['economy'] = np.where(bowl_season['overs'] > 0,
    bowl_season['total_runs_conceded'] / bowl_season['overs'], 0)
bowl_season['bowling_avg'] = np.where(bowl_season['total_wickets'] > 0,
    bowl_season['total_runs_conceded'] / bowl_season['total_wickets'], np.nan)

print(f"Bowling stats: {bowl_season.shape[0]} player-seasons, {bowl_season['bowler'].nunique()} unique players")
print(f"\nTop 5 wicket-takers (career):")
career_w = bowl_season.groupby('bowler')['total_wickets'].sum().sort_values(ascending=False)
for name, wkts in career_w.head(5).items():
    print(f"  {name}: {int(wkts)} wickets")

bowl_season.to_csv(f'{PROCESSED_DIR}/bowling_season_stats.csv', index=False)
bowl_season.to_parquet(f'{PROCESSED_DIR}/bowling_season_stats.parquet', index=False)
print(f"\n✅ Saved bowling_season_stats.csv and .parquet")

Bowling stats: 1948 player-seasons, 530 unique players

Top 5 wicket-takers (career):
  YS Chahal: 213 wickets
  DJ Bravo: 207 wickets
  PP Chawla: 201 wickets
  SP Narine: 200 wickets
  R Ashwin: 198 wickets

✅ Saved bowling_season_stats.csv and .parquet


### 4.3 Player of the Match Awards

In [8]:
potm = matches.groupby(['player_of_match', 'season']).size().reset_index(name='potm_awards')
potm_career = potm.groupby('player_of_match')['potm_awards'].sum().sort_values(ascending=False)
print(f"POTM records: {len(potm)} player-seasons")
print(f"\nTop 5 POTM winners (career):")
for name, awards in potm_career.head(5).items():
    print(f"  {name}: {int(awards)} awards")

POTM records: 760 player-seasons

Top 5 POTM winners (career):
  AB de Villiers: 25 awards
  CH Gayle: 22 awards
  RG Sharma: 19 awards
  DA Warner: 18 awards
  V Kohli: 18 awards


## 5. Name Matching & Feature Engineering

### 5.1 The Name Matching Challenge

The most critical data wrangling challenge: **auction data uses full names** ("Virat Kohli") while **ball-by-ball data uses initials** ("V Kohli"). Without resolving this, we cannot link auction prices to on-field performance.

**Solution:** A two-step matching strategy:
1. **ID map lookup** — Use the 620 pre-built name mappings from `player_id_map.csv`
2. **Direct match** — For names that happen to be identical across datasets

This achieves ~84% matching on the full auction dataset. The remaining ~16% are genuinely uncapped players entering their first IPL auction — correctly showing zero prior IPL stats.

In [9]:
# Build name mapping
name_map = dict(zip(id_map['auction_name'], id_map['cricsheet_name']))
all_bbb_names = set(bat_season['batter'].unique()) | set(bowl_season['bowler'].unique())

# Filter to sold players only (these are what we model)
sold_df = auction[auction['status'] == 'sold'].copy()

def resolve_name(auction_name):
    """Two-step name resolution: ID map first, then direct match."""
    mapped = name_map.get(auction_name)
    if mapped and mapped in all_bbb_names:
        return mapped
    if auction_name in all_bbb_names:
        return auction_name
    return auction_name  # Return as-is for uncapped players

sold_df['bbb_name'] = sold_df['player'].apply(resolve_name)

# Report match rates
matched = sum(1 for n in sold_df['bbb_name'] if n in all_bbb_names)
has_prior_stats = 0
for _, row in sold_df.iterrows():
    name = row['bbb_name']
    year = row['auction_year']
    pb = bat_season[(bat_season['batter'] == name) & (bat_season['season'] < year)]
    pw = bowl_season[(bowl_season['bowler'] == name) & (bowl_season['season'] < year)]
    if len(pb) > 0 or len(pw) > 0:
        has_prior_stats += 1

print(f"Name resolution results:")
print(f"  Total sold auction records: {len(sold_df)}")
print(f"  Names found in ball-by-ball data: {matched}/{len(sold_df)} ({matched/len(sold_df)*100:.0f}%)")
print(f"  Players with prior IPL stats: {has_prior_stats}/{len(sold_df)} ({has_prior_stats/len(sold_df)*100:.0f}%)")
print(f"  Uncapped/new to IPL: {len(sold_df) - has_prior_stats}/{len(sold_df)} ({(len(sold_df)-has_prior_stats)/len(sold_df)*100:.0f}%)")
print(f"\nNote: Players matched in ball-by-ball but with 0 prior stats were bought")
print(f"before their first IPL season — correctly showing zero historical performance.")

Name resolution results:
  Total sold auction records: 1279
  Names found in ball-by-ball data: 1074/1279 (84%)
  Players with prior IPL stats: 784/1279 (61%)
  Uncapped/new to IPL: 495/1279 (39%)

Note: Players matched in ball-by-ball but with 0 prior stats were bought
before their first IPL season — correctly showing zero historical performance.


### 5.2 Feature Engineering

For each auction record, we compute features from **prior seasons only** (seasons before the auction year) to avoid data leakage. This mirrors the real-world scenario: at the time of an auction, teams can only evaluate a player based on their past performance.

**Features computed (22 total):**

| Category | Features |
|----------|----------|
| **Batting** | career_runs, career_balls_faced, career_batting_sr, career_batting_avg, career_fours, career_sixes, career_boundary_pct, ipl_seasons_batted, career_bat_matches, recent_runs (last 2 seasons), recent_batting_sr |
| **Bowling** | career_wickets, career_bowl_balls, career_economy, career_bowling_avg, ipl_seasons_bowled, recent_wickets, recent_economy |
| **General** | total_ipl_seasons, total_potm, price_ratio (sold/base), has_ipl_experience |

In [10]:
# Build features for every sold auction record
features_rows = []
for _, row in sold_df.iterrows():
    name = row['bbb_name']
    year = row['auction_year']
    
    # Prior batting performance
    pb = bat_season[(bat_season['batter'] == name) & (bat_season['season'] < year)]
    pw = bowl_season[(bowl_season['bowler'] == name) & (bowl_season['season'] < year)]
    pp = potm[(potm['player_of_match'] == name) & (potm['season'] < year)]
    
    f = {}
    
    # === BATTING FEATURES ===
    if len(pb) > 0:
        f['career_runs'] = int(pb['total_runs'].sum())
        f['career_balls_faced'] = int(pb['total_balls'].sum())
        f['career_batting_sr'] = round(f['career_runs'] / f['career_balls_faced'] * 100, 2) if f['career_balls_faced'] > 0 else 0
        f['career_batting_avg'] = round(pb['batting_avg'].mean(), 2)
        f['career_fours'] = int(pb['total_fours'].sum())
        f['career_sixes'] = int(pb['total_sixes'].sum())
        f['career_boundary_pct'] = round(pb['boundary_pct'].mean(), 2)
        f['ipl_seasons_batted'] = int(len(pb))
        f['career_bat_matches'] = int(pb['matches'].sum())
        # Recent form (last 2 seasons)
        rb = pb[pb['season'] >= year - 2]
        f['recent_runs'] = int(rb['total_runs'].sum()) if len(rb) > 0 else 0
        rb_balls = rb['total_balls'].sum() if len(rb) > 0 else 0
        f['recent_batting_sr'] = round(rb['total_runs'].sum() / rb_balls * 100, 2) if rb_balls > 0 else 0
    else:
        f.update({k: 0 for k in ['career_runs', 'career_balls_faced', 'career_batting_sr',
            'career_batting_avg', 'career_fours', 'career_sixes', 'career_boundary_pct',
            'ipl_seasons_batted', 'career_bat_matches', 'recent_runs', 'recent_batting_sr']})
    
    # === BOWLING FEATURES ===
    if len(pw) > 0:
        f['career_wickets'] = int(pw['total_wickets'].sum())
        f['career_bowl_balls'] = int(pw['total_balls'].sum())
        total_overs = pw['overs'].sum()
        f['career_economy'] = round(pw['total_runs_conceded'].sum() / total_overs, 2) if total_overs > 0 else 0
        f['career_bowling_avg'] = round(pw.loc[pw['bowling_avg'].notna(), 'bowling_avg'].mean(), 2) if pw['bowling_avg'].notna().any() else 0
        f['ipl_seasons_bowled'] = int(len(pw))
        rw = pw[pw['season'] >= year - 2]
        f['recent_wickets'] = int(rw['total_wickets'].sum()) if len(rw) > 0 else 0
        rw_overs = rw['overs'].sum() if len(rw) > 0 else 0
        f['recent_economy'] = round(rw['total_runs_conceded'].sum() / rw_overs, 2) if rw_overs > 0 else 0
    else:
        f.update({k: 0 for k in ['career_wickets', 'career_bowl_balls', 'career_economy',
            'career_bowling_avg', 'ipl_seasons_bowled', 'recent_wickets', 'recent_economy']})
    
    # === GENERAL FEATURES ===
    f['total_ipl_seasons'] = max(f.get('ipl_seasons_batted', 0), f.get('ipl_seasons_bowled', 0))
    f['total_potm'] = int(pp['potm_awards'].sum()) if len(pp) > 0 else 0
    f['price_ratio'] = round(row['sold_price_cr'] / row['base_price_cr'], 2) if row['base_price_cr'] > 0 else np.nan
    f['has_ipl_experience'] = 1 if f['total_ipl_seasons'] > 0 else 0
    
    features_rows.append(f)

features_df = pd.DataFrame(features_rows)
ml_dataset = pd.concat([sold_df.reset_index(drop=True), features_df], axis=1)

print(f"ML-ready dataset: {ml_dataset.shape[0]} rows × {ml_dataset.shape[1]} columns")
print(f"\nExperience breakdown:")
print(f"  With prior IPL stats: {(ml_dataset['has_ipl_experience']==1).sum()} ({(ml_dataset['has_ipl_experience']==1).mean()*100:.0f}%)")
print(f"  Uncapped/new: {(ml_dataset['has_ipl_experience']==0).sum()} ({(ml_dataset['has_ipl_experience']==0).mean()*100:.0f}%)")

# Summary stats for experienced players
exp = ml_dataset[ml_dataset['has_ipl_experience'] == 1]
print(f"\nFeature summary (experienced players only, n={len(exp)}):")
print(exp[['sold_price_cr', 'career_runs', 'career_batting_sr', 'career_wickets',
           'career_economy', 'total_ipl_seasons', 'total_potm', 'price_ratio']].describe().round(2))

ML-ready dataset: 1279 rows × 31 columns

Experience breakdown:
  With prior IPL stats: 784 (61%)
  Uncapped/new: 495 (39%)

Feature summary (experienced players only, n=784):
       sold_price_cr  career_runs  career_batting_sr  career_wickets  career_economy  total_ipl_seasons  total_potm  \
count         784.00       784.00             784.00          784.00          784.00             784.00      784.00   
mean            3.53       536.43             100.44           24.55            6.53               4.62        1.87   
std             4.31       970.21              44.11           38.39            4.08               3.42        3.03   
min             0.10         0.00               0.00            0.00            0.00               1.00        0.00   
25%             0.60        20.00              78.10            0.00            5.73               2.00        0.00   
50%             2.00       106.00             113.60            7.00            7.79               4.00       

### 5.3 Validation Spot-Checks

In [11]:
# Spot-check known players to validate feature engineering
check_players = ['Virat Kohli', 'Jasprit Bumrah', 'MS Dhoni', 'Rashid Khan']
for p in check_players:
    rows = ml_dataset[ml_dataset['player'] == p].sort_values('auction_year')
    if len(rows) > 0:
        latest = rows.iloc[-1]
        print(f"\n{p} (auction {int(latest['auction_year'])}, sold ₹{latest['sold_price_cr']:.2f} Cr):")
        print(f"  Career runs: {int(latest['career_runs'])}, SR: {latest['career_batting_sr']:.1f}")
        print(f"  Career wickets: {int(latest['career_wickets'])}, Econ: {latest['career_economy']:.2f}")
        print(f"  IPL seasons: {int(latest['total_ipl_seasons'])}, POTM: {int(latest['total_potm'])}")
        print(f"  Price ratio: {latest['price_ratio']:.1f}x base price")


Virat Kohli (auction 2025, sold ₹21.00 Cr):
  Career runs: 8014, SR: 128.5
  Career wickets: 5, Econ: 8.43
  IPL seasons: 17, POTM: 18
  Price ratio: nanx base price

Jasprit Bumrah (auction 2025, sold ₹18.00 Cr):
  Career runs: 68, SR: 85.0
  Career wickets: 182, Econ: 7.23
  IPL seasons: 11, POTM: 10
  Price ratio: nanx base price

MS Dhoni (auction 2025, sold ₹4.00 Cr):
  Career runs: 5243, SR: 132.8
  Career wickets: 0, Econ: 0.00
  IPL seasons: 17, POTM: 17
  Price ratio: nanx base price

Rashid Khan (auction 2025, sold ₹18.00 Cr):
  Career runs: 545, SR: 153.5
  Career wickets: 157, Econ: 6.91
  IPL seasons: 8, POTM: 12
  Price ratio: nanx base price


## 6. Final Dataset & Summary

In [12]:
# Save final ML dataset
ml_dataset.to_csv(f'{PROCESSED_DIR}/ml_auction_dataset.csv', index=False)
ml_dataset.to_parquet(f'{PROCESSED_DIR}/ml_auction_dataset.parquet', index=False)

print("=" * 60)
print("WEEK 1 DATA FOUNDATION — COMPLETE")
print("=" * 60)
print(f"\nProcessed files:")
for f in sorted(os.listdir(PROCESSED_DIR)):
    size = os.path.getsize(f'{PROCESSED_DIR}/{f}') / 1024
    print(f"  {f} ({size:.0f} KB)")

print(f"\nDataset dimensions:")
print(f"  Auction unified:     {auction.shape[0]:>5} rows × {auction.shape[1]:>2} cols (all auction records 2013–2025)")
print(f"  Batting stats:       {bat_season.shape[0]:>5} rows × {bat_season.shape[1]:>2} cols (player-season aggregates)")
print(f"  Bowling stats:       {bowl_season.shape[0]:>5} rows × {bowl_season.shape[1]:>2} cols (player-season aggregates)")
print(f"  ML auction dataset:  {ml_dataset.shape[0]:>5} rows × {ml_dataset.shape[1]:>2} cols (sold records + 22 features)")

print(f"\nReady for Week 2: ML model training on {(ml_dataset['has_ipl_experience']==1).sum()} experienced player records.")

WEEK 1 DATA FOUNDATION — COMPLETE

Processed files:
  auction_unified.csv (78 KB)
  auction_unified.parquet (19 KB)
  batting_season_stats.csv (175 KB)
  batting_season_stats.parquet (67 KB)
  bowling_season_stats.csv (124 KB)
  bowling_season_stats.parquet (47 KB)
  ml_auction_dataset.csv (181 KB)
  ml_auction_dataset.parquet (89 KB)

Dataset dimensions:
  Auction unified:      1279 rows ×  8 cols (all auction records 2013–2025)
  Batting stats:        2617 rows × 11 cols (player-season aggregates)
  Bowling stats:        1948 rows ×  9 cols (player-season aggregates)
  ML auction dataset:   1279 rows × 31 cols (sold records + 22 features)

Ready for Week 2: ML model training on 784 experienced player records.


## 7. Next Steps (Week 2)

1. **Exploratory Data Analysis** — Distribution of auction prices, correlation analysis, feature importance
2. **ML Model Training** — Random Forest, XGBoost, and Linear Regression to predict sold_price_cr
3. **Feature Selection** — Identify which performance metrics most influence auction value
4. **Model Evaluation** — Cross-validation, RMSE, R² metrics

---
*Notebook generated as part of the IPL Auction Optimizer project.*  
*Demonstrates: Data wrangling (BUSAN302), Feature engineering (BUSAN300)*